# Creating Stationary ML-Ready Dataset

This notebook takes the raw merged dataset that we created in the last notebook (`raw_data_prediction_dataset.csv`) and transforms all independent variable columns into stationary form for better use in LSTM modeling (timeseries modeling). The target columns (`_logreturn_target`, `_volatility_target`) are left untouched as they should already be in the correct format.

**Transformations applied by column group:**
- **Price-like columns** (ETFs, indices, commodities): Log returns — `log(P_t / P_{t-1})`
- **Yield columns**: First difference — `change_yield` (already in % terms, so % change is not meaningful)
- **Quarterly macro columns**: QoQ % change — `pct_change(63)` (~63 trading days per quarter)
- **Monthly macro columns**: MoM % change — `pct_change(21)` (~21 trading days per month)
- **Weekly macro columns**: WoW % change — `pct_change(5)` (~5 trading days per week)

**Output:** `stationary_prediction_dataset.csv`

**Note:** This notebook is a companion to Notebook 4. Run Notebook 4 first to generate the raw dataset but we found the non-stationary data was not appropriate for our predictions

## Import Libraries

In [33]:
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Load Raw Dataset

We need to load the raw prediction dataset made in Notebook 4. This contains raw prices and levels for all features, plus the pre-computed target labels.

In [34]:
df = pd.read_csv('raw_data_prediction_dataset.csv', parse_dates=['date'])
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

Shape: (24955, 93)
Date range: 1927-12-30 00:00:00 to 2026-03-18 00:00:00


,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,XLP_volatility_target,XLY_volatility_target,XLRE_volatility_target,BIL_volatility_target,IEF_volatility_target,TLT_volatility_target,LQD_volatility_target,HYG_volatility_target,TIP_volatility_target,GLD_volatility_target
0,1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Define Column Groups

We need to make explicit column lists for each transformation type (price, percent, QoQ, etc). Each column should appear in exactly one group.
Target columns are excluded from all transformations by identifying them with '_target'.

In [35]:
# Columns to never transform
target_cols = [col for col in df.columns if '_target' in col]
skip_cols = ['date'] + target_cols

print(f"Target columns ({len(target_cols)}):")
print(target_cols)

Target columns (34):
['XLF_logreturn_target', 'XLK_logreturn_target', 'XLU_logreturn_target', 'XLV_logreturn_target', 'XLE_logreturn_target', 'XLI_logreturn_target', 'XLB_logreturn_target', 'XLP_logreturn_target', 'XLY_logreturn_target', 'XLRE_logreturn_target', 'BIL_logreturn_target', 'IEF_logreturn_target', 'TLT_logreturn_target', 'LQD_logreturn_target', 'HYG_logreturn_target', 'TIP_logreturn_target', 'GLD_logreturn_target', 'XLF_volatility_target', 'XLK_volatility_target', 'XLU_volatility_target', 'XLV_volatility_target', 'XLE_volatility_target', 'XLI_volatility_target', 'XLB_volatility_target', 'XLP_volatility_target', 'XLY_volatility_target', 'XLRE_volatility_target', 'BIL_volatility_target', 'IEF_volatility_target', 'TLT_volatility_target', 'LQD_volatility_target', 'HYG_volatility_target', 'TIP_volatility_target', 'GLD_volatility_target']


Alright, let's start with all of the price columns and transform them to logreturns. This should make the data stationary (avoiding drift). We can find log returns by the transformation: log return = log(P_t / P_{t-1}). Of note, ^MOVE, ^VXN, and ^VIX are volatility indices — log returns are used here but first difference is also defensible. If model performance is still poor, we may want to revist this.

In [36]:
# Price Columns
price_cols = [
    # ETFs
    'BIL', 'BND', 'GLD', 'HYG', 'IEF', 'IWM', 'LQD', 'QQQ', 'SPY',
    'TIP', 'TLT', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE',
    'XLU', 'XLV', 'XLY', 'SHY',
    # Commodity futures
    'CL=F', 'GC=F', 'HG=F', 'NG=F',
    # Equity indices
    '^FTSE', '^GSPC', '^HSI',
    # Volatility indices (log returns — see note above)
    '^MOVE', '^VIX', '^VXN'
]

Next, let's identify the yield columns (interest rates). Here we are looking at first differencing which is just the change (delta) in the yield. So the transformation: first difference = delta yield should be quite straight forward. 

In [37]:
# Yield Columns
yield_cols = [
    'one_month_yield', 'three_month_yield', 'six_month_yield',
    'one_year_yield', 'two_year_yield', 'five_year_yield',
    'ten_year_yield', 'thirty_year_yield',
    '^TNX'  # 10-year Treasury yield from yfinance — same rationale
]

Alright, now for macro-economic data we need to calculate the QoQ changes. We are going to have to do this based on the assumption that we are looking at 63 trading days in advance (1 quarter). This actually may need to be adjusted because it is just taking 252/4 which is the average. The transformation is QoQ % change = pct_change(63). 

In [38]:
# Quarterly macro columns
qoq_cols = [
    'nominal_GDP', 'real_GDP', 'debt_to_GDP', 'debt_interest',
    'consumer_price_index', 'core_PCI', 'personal_consumption_expenditure',
    'core_PCE', 'producer_price_index'
]

Do the same for Montly

In [39]:
# Monthly macro columns
mom_cols = [
    'unemployment_rate',
    'teenager_unemployment_rate', 'adult_unemployment_rate',
    'male_unemployment_rate', 'female_unemployment_rate',
    'average_duration_of_unemployment'
]

and Weekly

In [40]:
# Weekly macro columns
wow_cols = [
    'initial_jobless_claims',
    'continued_jobless_claims'
]

Alright, let's make sure we have every non-target column and that we didn't duplicate any

In [41]:
# Sanity check, let's use sets for convenience
all_transform_cols = set(price_cols + yield_cols + qoq_cols + mom_cols + wow_cols)
all_feature_cols = set(df.columns) - set(skip_cols)

unaccounted = all_feature_cols - all_transform_cols
duplicated = [
    col for col in (price_cols + yield_cols + qoq_cols + mom_cols + wow_cols)
    if (price_cols + yield_cols + qoq_cols + mom_cols + wow_cols).count(col) > 1
]

print(f"Unaccounted feature columns (need to be assigned to a group): {unaccounted}")
print(f"Duplicated columns (appear in more than one group): {duplicated}")
if not unaccounted and not duplicated:
    print("All feature columns are accounted for with no duplicates.")

Unaccounted feature columns (need to be assigned to a group): set()
Duplicated columns (appear in more than one group): []
All feature columns are accounted for with no duplicates.


## Apply Transformations

Okay, so all feature columns are accounted for and we didn't duplicate any so now all transformations need to be applied to a copy of the dataframe. We will keep the original df just in case for comparison. Target columns are copied over but should be unchanged.

In [42]:
# Note: We came back to add this after some data exploration (see below)
# CL=F went negative in April 2020 (COVID storage collapse) — clip to small positive floor
# before log transformation to avoid invalid log values. This is a known market anomaly specific to futures contracts and does not affect any target variables.
df['CL=F'] = df['CL=F'].clip(lower=0.01)

In [43]:
# we are going to start with the data and target columns only, then add transformed features one group at a time to keep it organized and easier to debug if needed
stationary_df = df[skip_cols].copy()

# Log returns for price columns
for col in price_cols:
    stationary_df[col] = np.log(df[col] / df[col].shift(1))

# First difference for yield columns
for col in yield_cols:
    stationary_df[col] = df[col].diff(1)

# QoQ % change for quarterly macro columns (63 trading days per quarter)
for col in qoq_cols:
    stationary_df[col] = df[col].pct_change(63)

# MoM % change for monthly macro columns (21 trading days per month)
for col in mom_cols:
    stationary_df[col] = df[col].pct_change(21)

# WoW % change for weekly macro columns (5 trading days per week)
for col in wow_cols:
    stationary_df[col] = df[col].pct_change(5)

# Let's take a look at the dataframe (going to copy and drop NaNs for the sample to avoid showing all the NaN rows at the top due to differencing and pct_change)
no_nans_df = stationary_df.dropna()

print(f"Stationary DataFrame shape: {no_nans_df.shape}")
no_nans_df.tail(5)

Stationary DataFrame shape: (50, 93)


,date,XLF_logreturn_target,XLK_logreturn_target,XLU_logreturn_target,XLV_logreturn_target,XLE_logreturn_target,XLI_logreturn_target,XLB_logreturn_target,XLP_logreturn_target,XLY_logreturn_target,...,core_PCE,producer_price_index,unemployment_rate,teenager_unemployment_rate,adult_unemployment_rate,male_unemployment_rate,female_unemployment_rate,average_duration_of_unemployment,initial_jobless_claims,continued_jobless_claims
24423,2024-02-26,0.162889,0.191096,0.685291,-0.059130,0.262400,0.156980,0.241792,0.167632,-0.132558,...,0.009662,0.007180,0.054054,0.177570,-0.027778,-0.02500,0.147059,0.000000,0.054187,-0.003895
24677,2025-02-19,-0.051180,-0.136069,0.151468,-0.317724,-0.353252,0.153299,-0.107513,0.072306,-0.173722,...,0.009543,0.012196,0.050000,0.101695,0.000000,0.02439,0.025000,-0.027273,0.041860,-0.007523
24678,2025-02-20,-0.072047,-0.199194,0.078319,-0.433807,-0.463981,0.111421,-0.151779,0.041271,-0.221028,...,0.009543,0.012196,0.050000,0.101695,0.000000,0.02439,0.025000,-0.027273,0.041860,-0.007523
24679,2025-02-21,-0.026694,-0.088289,0.019810,-0.443718,-0.395600,0.199170,-0.082168,-0.021931,-0.097238,...,0.009543,0.012196,0.050000,0.101695,0.000000,0.02439,0.025000,-0.027273,0.041860,-0.007523
24680,2025-02-24,-0.058356,-0.075025,0.087447,-0.482534,-0.383774,0.204264,-0.085550,-0.019348,-0.118296,...,0.009543,0.012196,0.050000,0.101695,0.000000,0.02439,0.025000,-0.027273,0.084821,0.024364


Interesting that we are getting a runtime error here, there must be a negative value or zero (which would be inaccurate) so let's see if we can find it.

In [44]:
for col in price_cols:
    if (df[col] <= 0).any():
        print(col, (df[col] <= 0).sum(), 'non-positive values')

Crude oil! This actually isn't an artifact because it went negative during COVID when storage capacity ran out. We may want to consider clipping this before we transform. we can clip this so it's lowed value is 0.01. We will do this right before the transformation block and indicate it.

Alright, we didn't really pick and choose the columns that we want and when we arbitrarily drop NaN rows, we only have 50 rows, which is very short term. This is something we will have to pay attention to.

## Sanity Checks

Based on what we just saw, let's verify the transformations look correct before exporting.

In [45]:
# Check NaN counts — expect leading NaNs from the shift/pct_change operations
# QoQ columns will have the most (63 NaNs), WoW the fewest (5 NaNs)
# Target columns will have NaNs at the tail (last 63 rows) — this is correct
stationary_df.isna().sum()

date                                    0
XLF_logreturn_target                18339
XLK_logreturn_target                18339
XLU_logreturn_target                18339
XLV_logreturn_target                18339
                                    ...  
male_unemployment_rate               8518
female_unemployment_rate             8518
average_duration_of_unemployment     8518
initial_jobless_claims               9780
continued_jobless_claims             9780
Length: 93, dtype: int64

In [46]:
# SPY log returns should be small values centered around zero
# Large outliers (e.g. >0.20) should correspond to known market events (COVID crash, GFC)
print("SPY log return stats:")
print(stationary_df['SPY'].describe())
print(f"\nLargest single-day moves:")
print(stationary_df.nsmallest(5, 'SPY')[['date', 'SPY']])
print(stationary_df.nlargest(5, 'SPY')[['date', 'SPY']])

SPY log return stats:
count    8128.000000
mean        0.000424
std         0.011689
min        -0.115886
25%        -0.004313
50%         0.000698
75%         0.005916
max         0.135577
Name: SPY, dtype: float64

Largest single-day moves:
            date       SPY
23403 2020-03-16 -0.115886
20459 2008-10-15 -0.103638
23401 2020-03-12 -0.100569
20492 2008-12-01 -0.092750
20447 2008-09-29 -0.081602
            date       SPY
20457 2008-10-13  0.135577
20468 2008-10-28  0.110517
24712 2025-04-09  0.099863
23409 2020-03-24  0.086731
23402 2020-03-13  0.082028


In [47]:
# ten_year_yield first difference should be small daily basis point moves
print("10Y yield first difference stats:")
print(stationary_df['ten_year_yield'].describe())

10Y yield first difference stats:
count    16443.000000
mean         0.000010
std          0.064063
min         -0.750000
25%         -0.030000
50%          0.000000
75%          0.030000
max          0.650000
Name: ten_year_yield, dtype: float64


In [48]:
# Confirm target columns are unchanged from the raw dataset
sample_target = target_cols[0]
unchanged = df[sample_target].equals(stationary_df[sample_target])
print(f"Target column '{sample_target}' unchanged: {unchanged}")

Target column 'XLF_logreturn_target' unchanged: True


## Visual Sanity Check

Finally, let's plot a few transformed series to confirm they look stationary (mean-reverting, no clear trend).

In [53]:
# Plot SPY log returns — should look like white noise around zero
spy_df = stationary_df[['date', 'SPY']].dropna()

alt.Chart(spy_df).mark_line(strokeWidth=0.8, color='blue').encode(
    x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)),
    y=alt.Y('SPY:Q', axis=alt.Axis(title='Log Return', grid=False))
).properties(title='SPY Daily Log Returns', height=300, width=900).configure_view(strokeWidth=0)

alt.Chart(...)

In [54]:
# Plot 10Y yield first difference — should show basis point moves, no trend
yield_df = stationary_df[['date', 'ten_year_yield']].dropna()

alt.Chart(yield_df).mark_line(strokeWidth=0.8, color='red').encode(
    x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)),
    y=alt.Y('ten_year_yield:Q', axis=alt.Axis(title='Δ Yield (pp)', grid=False))
).properties(title='10Y Treasury Yield — First Difference', height=300, width=900).configure_view(strokeWidth=0)

alt.Chart(...)

In [55]:
# Plot nominal GDP QoQ % change — should look like a slow-moving series
# with a sharp dip visible during the GFC (2008-09) and COVID (2020)
gdp_df = stationary_df[['date', 'nominal_GDP']].dropna()

alt.Chart(gdp_df).mark_line(strokeWidth=1, color='green').encode(
    x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)),
    y=alt.Y('nominal_GDP:Q', axis=alt.Axis(title='QoQ % Change', grid=False))
).properties(title='Nominal GDP — QoQ % Change', height=300, width=900).configure_view(strokeWidth=0)

alt.Chart(...)

Great, all of our data looks pretty stationary (although heteroskedastic) with high variance clustering during periods of volatility. We will have to see if the LSTM architecture accounts for this or if it treats everything equally. Standard LSTMs assume constant variance if I am not mistaken, which could be an issue. This may be why GARCH is the prefered model. 

## Export Data

Alright, things look good so we will export the stationary dataset. This dataset still contains leading NaNs from the transformation window and trailing NaNs in the target columns. These will be handled in the modeling notebook with an
explicit date filter (e.g. >= 2009-01-01) and the existing buffer/split logic.

In [52]:
stationary_df.to_csv('stationary_prediction_dataset.csv', index=False)
print(f"Exported stationary_prediction_dataset.csv — shape: {stationary_df.shape}")

Exported stationary_prediction_dataset.csv — shape: (24955, 93)
